# 06 — Performance Analysis

Loads benchmark JSON artifacts, computes speedup / throughput with
mean ± std, and plots comparisons.

**Parallel efficiency is deliberately not computed** (no valid scaling
study — see docs/EXPERIMENT_DESIGN.md §3).

In [ ]:
import sys, json, math
from pathlib import Path
import pandas as pd
root = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(root / "src"))

In [ ]:
from roadvision.benchmarks.metrics import speedup, throughput, mean_std, parallel_efficiency
print("speedup(10s CPU, 2s GPU):", speedup(10, 2))
try:
    parallel_efficiency(speedup_value=4.0, core_counts=[4])
except Exception as e:
    print("efficiency correctly refused:", type(e).__name__)

In [ ]:
bench_dir = root / "results" / "benchmarks"
files = sorted(bench_dir.glob("*.json")) if bench_dir.exists() else []
print("artifacts:", [f.name for f in files])
for f in files:
    if "environment" in f.name or "feasibility" in f.name:
        print("\n==", f.name)
        print(f.read_text()[:2000])

In [ ]:
# Build a comparison table from saved epoch histories (Experiment D).
# Expected inputs: history_cpu.json / history_gpu.json from the SAME Colab runtime.
rows = []
for device in ("cpu", "gpu"):
    p = root / "results" / "benchmarks" / f"history_{device}.json"
    if p.exists():
        h = json.loads(p.read_text())
        for ep in h["epochs"]:
            t = ep["train"]
            rows.append({
                "device": device,
                "epoch": ep["epoch"],
                "train_time_s": t["end_to_end"],
                "throughput": t["throughput_img_s"],
                "loss": t["loss_mean"],
                "peak_rss_mb": t.get("peak_cpu_rss_mb"),
                "peak_gpu_alloc_mb": t.get("peak_gpu_alloc_mb"),
            })
    else:
        print("missing:", p.name, "(run Experiment D first)")
df = pd.DataFrame(rows)
if not df.empty:
    display(df)
else:
    print("No history files yet — no numbers will be invented.")

In [ ]:
# Speedup with mean ± std across epochs (only when both devices present)
if not df.empty and set(df["device"]) == {"cpu", "gpu"}:
    cpu_t = df[df.device=="cpu"]["train_time_s"]
    gpu_t = df[df.device=="gpu"]["train_time_s"]
    merged = cpu_t.reset_index(drop=True).to_frame("cpu").join(
        gpu_t.reset_index(drop=True).to_frame("gpu"))
    merged["speedup"] = merged["cpu"] / merged["gpu"]
    print(merged)
    m, s = mean_std(merged["speedup"].tolist())
    print(f"speedup = {m:.3f} ± {s:.3f}  (n={len(merged)})")
else:
    print("Both device histories required for speedup — nothing plotted.")

In [ ]:
# Plots (only render what exists)
from roadvision.visualization.plots import plot_device_comparison
if not df.empty and set(df["device"]) == {"cpu", "gpu"}:
    eps = sorted(df["epoch"].unique())
    cpu_times = [float(df[(df.device=="cpu") & (df.epoch==e)]["train_time_s"].iloc[0]) for e in eps]
    gpu_times = [float(df[(df.device=="gpu") & (df.epoch==e)]["train_time_s"].iloc[0]) for e in eps]
    out = plot_device_comparison(
        [f"epoch {e}" for e in eps], cpu_times, gpu_times,
        ylabel="train time (s)", title="Experiment D: CPU vs GPU (same runtime)",
        save_path=root / "results" / "visualizations" / "expD_train_time.png")
    print("saved:", out)
else:
    print("No paired data — plot skipped (would be empty/fabricated).")